# Prediction notebook template

Upload THIS notebook (card 1) with your trained model (card 2). The platform runs it top to bottom in a sandbox with **no internet** and **no `train.csv`**: it must only load the model, build the same features as in training, and predict.

* Train in your own notebook on your machine (not uploaded), with the platform's library versions (`pip install -r requirements-image.txt`).
* On the platform the test file has **no ID column and its rows are shuffled**. Write one prediction per row, in the same order, under the column `prediction`. Track 2: probability of `fraud` (0 to 1).

Example training code for your own notebook (it saves `model.pkl` for card 2):
```python
train = pd.read_csv("train.csv")
label = "energy_usage"   # Track 2: "fraud"
X = make_features(train)
features = [c for c in X.columns if c not in ("id", label)]
text = [c for c in features if X[c].dtype == object]
encode = ColumnTransformer([("text", OneHotEncoder(handle_unknown="ignore", sparse_output=False), text)], remainder="passthrough")
model = Pipeline([("encode", encode), ("model", HistGradientBoostingRegressor(random_state=42))])  # Track 2: ...Classifier
model.fit(X[features], train[label])
joblib.dump((model, features), "model.pkl")
```

In [ ]:
import os
import joblib
import pandas as pd

TEST_PATH = os.environ.get("DATATHON_INPUT_PATH", "test.csv")
OUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")

In [ ]:
def make_features(df):
    """The SAME feature engineering as in your training notebook (copy it from there)."""
    df = df.copy()
    return df

In [ ]:
model, features = joblib.load("model.pkl")  # uploaded in card 2, next to this notebook
X_test = make_features(pd.read_csv(TEST_PATH))[features]
if hasattr(model, "predict_proba"):  # Track 2: probability of fraud
    preds = model.predict_proba(X_test)[:, 1]
else:  # Track 1
    preds = model.predict(X_test)

In [ ]:
pd.DataFrame({"prediction": preds}).to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(preds)} predictions to {OUTPUT_PATH}")